# 01 — TCGA Mutation Frequency Pipeline

Downloads masked somatic mutation (MAF) files from the GDC API for each configured TCGA
cancer type and computes, per gene, the fraction of samples in which it is mutated.

This notebook is a runnable, cell-by-cell version of `scripts/run_mutation_frequency_pipeline.py`
— it imports the same `src/` modules, so there's no logic duplicated between the notebook and
the package.

**Requires:** network access to the GDC API. Running the full loop over all 24 cancer types can
take a while.


In [ ]:
import sys
import os
from pathlib import Path

# Notebooks live in notebooks/, but config.py's paths (e.g. "data/...") are
# relative to the repo root — chdir there so those paths resolve correctly
# regardless of where Jupyter's working directory starts out.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd

import config
from src.gdc_client import get_maf_file_ids, download_maf_files
from src.mutation_frequency import compute_mutation_frequency
from src.visualize import (
    plot_mutation_frequency_distribution,
    plot_mutation_frequency_by_cancer,
    plot_top_genes_by_cancer,
)

## Configuration

Cancer types, GDC endpoints, and output paths all come from `config.py` at the repo root, so
this notebook and the equivalent script always stay in sync.

In [ ]:
config.DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Cancer types: {len(config.CANCER_TYPES)}")
print(f"Output file: {config.MUTATION_FREQUENCY_FILE}")

## Download MAFs and compute mutation frequency per cancer type

For each cancer type: look up its MAF file IDs on GDC, download any not already cached
locally, then compute per-gene mutation frequency for that cancer type.

In [ ]:
all_results = []

for cancer in config.CANCER_TYPES:
    try:
        print(f"\n[PROCESSING] {cancer}")

        cancer_dir = config.TCGA_MAF_DIR / cancer
        existing_files = list(cancer_dir.rglob("*.maf*")) if cancer_dir.exists() else []

        if existing_files:
            print(f"[SKIP] {len(existing_files)} MAF files already present for {cancer} — skipping download")
        else:
            file_ids = get_maf_file_ids(cancer)
            print(f"[INFO] {len(file_ids)} files found")
            download_maf_files(file_ids, cancer)

        df = compute_mutation_frequency(cancer_dir)

        if df is None or df.empty:
            print(f"[WARNING] No data for {cancer}")
            continue

        df["cancer_type"] = cancer
        print(f"[INFO] {cancer}: {df.shape}")
        all_results.append(df)

    except Exception as e:
        print(f"[ERROR] {cancer}: {e}")
        continue

if not all_results:
    raise RuntimeError("No mutation data collected")

mutation_df = pd.concat(all_results, ignore_index=True)
mutation_df.to_csv(config.MUTATION_FREQUENCY_FILE, index=False)

print(f"\n[DONE] Saved: {config.MUTATION_FREQUENCY_FILE}")
mutation_df.head()

## Explore the results

Reload the saved CSV and look at the overall shape of the data.

In [ ]:
df = pd.read_csv(config.MUTATION_FREQUENCY_FILE)
df.head()

### Overall mutation frequency distribution (log scale)

In [ ]:
plot_mutation_frequency_distribution(df)

### Mutation frequency distribution per cancer type (overlaid)

In [ ]:
plot_mutation_frequency_by_cancer(df)

### Top 10 most frequently mutated genes, per cancer type

In [ ]:
for cancer in df["cancer_type"].unique():
    plot_top_genes_by_cancer(df, cancer, top_n=10)